# Gradient Boosting - Hyperparameter Tuning

The previous notebook explored selected Gradient Boosting hyperparameters manually. Increasing tree depth produced the largest improvement in fraud detection performance, but deeper trees also showed diminishing returns and an increasing risk of overfitting.

This notebook extends the manual experiments by introducing systematic hyperparameter tuning.

The tuning process will:

- explore multiple Gradient Boosting hyperparameter configurations,
- use F1-score as the primary optimization metric,
- preserve the temporal structure of the transaction data during validation,
- keep the 2018 validation set separate from hyperparameter tuning,
- keep 2019 completely unseen for final model evaluation.

## Experimental Design

The available labeled transaction history is divided according to time:

- **2010–2017:** model development and hyperparameter tuning
- **2018:** validation and model comparison
- **2019:** final out-of-time test set

Hyperparameter tuning must use only information available within the 2010–2017 development period. The temporal order of transactions will be preserved during cross-validation to avoid training models on future observations and evaluating them on the past.

In [4]:
from finsight.database import (
    connect_to_database,
    test_database_connection,
)

from finsight.fraud_data import (
    BASE_FEATURES,
    download_yearly_data,
)

In [2]:
engine = connect_to_database()
test_database_connection(engine)

Connected!


In [5]:
features = BASE_FEATURES

yearly_data = download_yearly_data(
    engine,
    start_year=2010,
    end_year=2017,
    total_non_fraud_limit=300000,
    features=features
)

In [7]:
for year, data in yearly_data.items():
    print(
        year,
        data["X"].shape,
        data["y"].value_counts().to_dict()
    )

2010 (37148, 13) {False: 34575, True: 2573}
2011 (36048, 13) {False: 36011, True: 37}
2012 (37815, 13) {False: 36892, True: 923}
2013 (39124, 13) {False: 37787, True: 1337}
2014 (38803, 13) {False: 38139, True: 664}
2015 (40896, 13) {False: 38707, True: 2189}
2016 (41251, 13) {False: 38803, True: 2448}
2017 (39258, 13) {False: 39086, True: 172}
